# Load video and model
## Video:
- video_test: short videos, 3 folders (HTT, UTTQ, UTDD)
  - HTT: Duodenal -> use model htt.pt
  - UTTQ: Esophageal cancer -> use model thucquan.pt
  - UTDD: Gastric cancer - use model daday.pt
- video_CS: IGH videos from Hoang Long Clinic
  - HTT: Duodenal -> use model htt.pt
  - UTTQ: Esophageal cancer -> use model thucquan.pt
  - UTDD: Gastric cancer - use model daday.pt
- data_pk: 5 videos (inflammatory object) -> can use (3 models) or 5-class model

## YOLOv8 Model
- 3 models and classes:
  - htt.pt: 7_Loet_HTT
  - thucquan.pt: 2_Viem_thuc_quan, 5_Ung_thu_thuc_quan
  - daday.pt: 3_Viem_da_day_HP_am, 4_Viem_da_day_HP_duong, 6_Ung_thu_da_day
- 1 model (5 classes): 5-class-model.pt
  - 0: 2_Viem_thuc_quan
  - 1: 3_Viem_da_day_HP_am
  - 2: 5_Ung_thu_thuc_quan
  - 3: 6_Ung_thu_da_day
  - 4: 7_Loet_HTT

## Re-ID Model
- OSNet (Market1501-based): osnet_x0_25_endocv_30.pt

## Usage
- Mount Drive + Load model
- Load Video data
- Install requirements
- Import libraries
- File define (edit 'name=...' when change the video)
- Class Color, StrongSORT
- Class ObjectDetection (simple use)
- RUN
- Generate txt csv results

The final results will be in folder "/content/runs"

# Mount Drive + Load model

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load ReID model and 3 detection models

In [13]:
!cp -r /content/drive/MyDrive/torchreid_model/osnet_x0_25_endocv_30.pt /content/drive/MyDrive/ENDOCV/model_pt/model_yolo/daday.pt /content/drive/MyDrive/ENDOCV/model_pt/model_yolo/thucquan.pt /content/drive/MyDrive/ENDOCV/model_pt/model_yolo/htt.pt /content

Load ReID model and 5-class model

In [ ]:
!cp -r /content/drive/MyDrive/torchreid_model/osnet_x0_25_endocv_30.pt /content/drive/MyDrive/ENDOCV/model_pt/model_yolo/5-class-model.pt /content

# Load Video data (optional)

In [ ]:
!cp -r /content/drive/MyDrive/data_pk /content

In [ ]:
!cp -r /content/drive/MyDrive/video_CS /content

In [14]:
!cp -r /content/drive/MyDrive/ENDOCV/video_test /content

# Install requirements

In [1]:
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install ultralytics
!pip install boxmot==10.0.65

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 63.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 69.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 92.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 50.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 107.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Import lib

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from time import perf_counter, time
import cv2
import numpy as np
import torch
from boxmot import StrongSORT
from pathlib import Path
import sys
from datetime import datetime, timedelta
import pandas as pd
from google.colab.patches import cv2_imshow
import os
import csv

In [ ]:
!mkdir -p /content/runs

# 1.1 File define (video_test)

In [ ]:
def get_files(directory):
    files = []
    for filename in os.listdir(directory):
        # Lấy tên file mà không có phần mở rộng
        file_name_without_extension, _ = os.path.splitext(filename)
        files.append(file_name_without_extension)
    return files

directory_A = "/content/video_test/UTDD/"
directory_B = "/content/video_test/UTTQ/"
directory_C = "/content/video_test/HTT/"

files_A = get_files(directory_A)
files_B = get_files(directory_B)
files_C = get_files(directory_C)

vid_utdd_uttq_htt = [files_A, files_B,files_C]

print(vid_utdd_uttq_htt)


[['230517BVK016', '230420BVK077', '230320BVK020', '230412BVK059', '230718BVK089', '230620BVK023'], ['230411BVK106', '230413BVK007', '230630BVK060', '230411BVK107', '230411BVK004', '230411BVK104', '230630BVK059', '230407BVK095'], ['Da day 211207 CS1 02_Trim']]


### video_test:
- UTDD: ['230718BVK089', '230620BVK023', '230517BVK016', '230420BVK077', '230412BVK059', '230320BVK020']
- UTTQ: ['230411BVK106', '230630BVK059', '230411BVK004', '230411BVK104', '230630BVK060', '230411BVK107', '230407BVK095', '230413BVK007']
- HTT: ['Da day 211207 CS1 02_Trim']


In [ ]:
                    # Copy and paste video name
name = '230411BVK107'


if name in vid_utdd_uttq_htt[0]:  # Kiểm tra xem name có trong chiều 1 không
    test_vid = "/content/video_test/UTDD/" + name + ".mp4"
    model_weights = "/content/daday.pt"
elif name in vid_utdd_uttq_htt[1]:
    test_vid = "/content/video_test/UTTQ/" + name + ".mp4"
    model_weights = "/content/thucquan.pt"
else:
    test_vid = "/content/video_test/HTT/" + name + ".mp4"
    model_weights = "/content/htt.pt"


input_video_name = test_vid.split("/")[-2].split(".")[0] + '_' + test_vid.split("/")[-1].split(".")[0]


# Tạo từ điển ánh xạ giữa tên model_weights và model_classes
model_classes_dict = {
    "/content/daday.pt": ['Viem da day', 'Viem da day' , 'Ung thu da day'],
    "/content/thucquan.pt": ['Viem thuc quan', 'Ung thu thuc quan'],
    "/content/htt.pt": ['Loet HTT']
}

# Thiết lập model_classes từ từ điển, nếu không khớp thì trả về ['polyp', 'esophagael cancer']
model_classes = model_classes_dict.get(model_weights, ['polyp', 'esophagael cancer'])


print("Input Video Name:", input_video_name)
print("Model Classes:", model_classes)

Input Video Name: UTTQ_230411BVK107
Model Classes: ['Viem thuc quan', 'Ung thu thuc quan']


# 1.2 File define (video_CS)

In [ ]:
def get_files(directory):
    files = []
    for filename in os.listdir(directory):
        # Lấy tên file mà không có phần mở rộng
        file_name_without_extension, _ = os.path.splitext(filename)
        files.append(file_name_without_extension)
    return files

directory_A = "/content/video_CS/UTDD/"
directory_B = "/content/video_CS/UTTQ/"
directory_C = "/content/video_CS/HTT/"

files_A = get_files(directory_A)
files_B = get_files(directory_B)
files_C = get_files(directory_C)

vid_utdd_uttq_htt = [files_A, files_B,files_C]

print(vid_utdd_uttq_htt)


[['210518CS101', '230114CS2', '220324CS2', '210504CS205', '220702CS2'], ['220727CS201', '231103CS101', '231109CS101', '220318CS202', '231124CS101', '220922CS201'], []]


### video_CS:
- UTDD ['210504CS205', '220702CS2', '220324CS2', '210518CS101', '230114CS2'],
- UTTQ ['231103CS101', '220318CS202', '231124CS101', '231109CS101', '220922CS201', '220727CS201']
- HTT []

In [ ]:
# Copy and paste video name
name = '231103CS101'


if name in vid_utdd_uttq_htt[0]:  # Kiểm tra xem name có trong chiều 1 không
    test_vid = "/content/video_test/UTDD/" + name + ".mp4"
    model_weights = "/content/daday.pt"
elif name in vid_utdd_uttq_htt[1]:
    test_vid = "/content/video_test/UTTQ/" + name + ".mp4"
    model_weights = "/content/thucquan.pt"
else:
    test_vid = "/content/video_test/HTT/" + name + ".mp4"
    model_weights = "/content/htt.pt"

input_video_name = test_vid.split("/")[-2].split(".")[0] + '_' + test_vid.split("/")[-1].split(".")[0]


# Tạo từ điển ánh xạ giữa tên model_weights và model_classes
model_classes_dict = {
    "/content/daday.pt": ['Viem da day', 'Viem da day' , 'Ung thu da day'],
    "/content/thucquan.pt": ['Viem thuc quan', 'Ung thu thuc quan'],
    "/content/htt.pt": ['Loet HTT']

}

# Thiết lập model_classes từ từ điển, nếu không khớp thì trả về ['polyp', 'esophagael cancer']
model_classes = model_classes_dict.get(model_weights, ['polyp', 'esophagael cancer'])


print("Input Video Name:", input_video_name)
print("Model Classes:", model_classes)

Input Video Name: UTTQ_231103CS101
Model Classes: ['Viem thuc quan', 'Viem da day', 'Ung thu thuc quan', 'Ung thu da day', 'Loet HTT']


# 1.3 File define (data_pk)




In [ ]:
def get_files(directory):
    files = []
    for filename in os.listdir(directory):
        # Lấy tên file mà không có phần mở rộng
        file_name_without_extension, _ = os.path.splitext(filename)
        files.append(file_name_without_extension)
    return files

#directory_A = "/content/video_CS/UTDD/"
directory_B = "/content/data_pk/"
#directory_C = "/content/video_CS/HTT/"

#files_A = get_files(directory_A)
files_B = get_files(directory_B)
#files_C = get_files(directory_C)

vid_utdd_uttq_htt = [files_B]

print(vid_utdd_uttq_htt)


[['Da day 220111 CS1 05', 'Da day 200508 CS1 02', 'IGH AINN20 Tổng hợp timeframe video gui CNTT', 'Da day 200530 CS1 02', 'Da day 200512 CS1 01', 'Da day 200926 CS1 01']]


In [ ]:
# 'Da day 200926 CS1 01' DD TQ
# 'Da day 200508 CS1 02' DD HTT TQ
# 'Da day 200530 CS1 02' DD TQ HTT
# 'Da day 200512 CS1 01' DD
# 'Da day 220111 CS1 05' DD TQ HTT

name = 'Da day 200512 CS1 01'

if name in vid_utdd_uttq_htt[0]:  # Kiểm tra xem name có trong chiều 1 không
    test_vid = "/content/data_pk/" + name + ".mp4"
    model_weights = "/content/5-class-model.pt
"
#input_video_name = test_vid.split("/")[-2].split(".")[0] + '_' + test_vid.split("/")[-1].split(".")[0]
input_video_name = name


# Tạo từ điển ánh xạ giữa tên model_weights và model_classes
model_classes_dict = {
    "/content/5-class-model.pt": ['Viem thuc quan', 'Viem da day' ,'Ung thu thuc quan', 'Ung thu da day', 'Loet HTT']
}

# Thiết lập model_classes từ từ điển, nếu không khớp thì trả về ['polyp', 'esophagael cancer']
model_classes = model_classes_dict.get(model_weights, ['polyp', 'esophagael cancer'])


print("Input Video Name:", input_video_name)
print("Model Classes:", model_classes)

Input Video Name: Da day 200512 CS1 01
Model Classes: ['Viem da day', 'Viem da day', 'Ung thu da day']


# Class Color, Strongsort

In [ ]:
class Colors:
    def __init__(self, num_colors=80):
        self.num_colors = num_colors
        self.color_palette = self.generate_color_palette()


    def generate_color_palette(self):
        hsv_palette = np.zeros((self.num_colors, 1, 3), dtype=np.uint8)
        hsv_palette[:, 0, 0] = np.linspace(0, 180, self.num_colors, endpoint=False)
        hsv_palette[:, :, 1:] = 255
        bgr_palette = cv2.cvtColor(hsv_palette, cv2.COLOR_HSV2BGR)
        return bgr_palette.reshape(-1, 3)

    def __call__(self, class_id):
        color = tuple(map(int, self.color_palette[class_id]))
        return color

In [ ]:
import numpy as np

from boxmot.appearance.reid_auto_backend import ReidAutoBackend
from boxmot.motion.cmc import get_cmc_method
from boxmot.trackers.strongsort.sort.detection import Detection
from boxmot.trackers.strongsort.sort.tracker import Tracker
from boxmot.utils.matching import NearestNeighborDistanceMetric
from boxmot.utils.ops import xyxy2tlwh
from boxmot.utils import PerClassDecorator


class StrongSORT(object):
    def __init__(
        self,
        model_weights,
        device,
        fp16,
        per_class=False,
        max_dist=0.2,
        max_iou_dist=0.7,
        max_age=30,
        n_init=1,
        nn_budget=100,
        mc_lambda=0.995,
        ema_alpha=0.9,
    ):
        self.max_dist=0.95,
        self.max_iou_dist=0.95,
        self.max_age=300,
        self.per_class = per_class
        rab = ReidAutoBackend(
            weights=model_weights, device=device, half=fp16
        )
        self.model = rab.get_backend()
        self.tracker = Tracker(
            metric=NearestNeighborDistanceMetric("cosine", max_dist, nn_budget),
            max_iou_dist=max_iou_dist,
            max_age=max_age,
            n_init=n_init,
            mc_lambda=mc_lambda,
            ema_alpha=ema_alpha,
        )
        self.cmc = get_cmc_method('ecc')()

    @PerClassDecorator
    def update(self, dets, img, embs=None):
        assert isinstance(
            dets, np.ndarray
        ), f"Unsupported 'dets' input format '{type(dets)}', valid format is np.ndarray"
        assert isinstance(
            img, np.ndarray
        ), f"Unsupported 'img' input format '{type(img)}', valid format is np.ndarray"
        assert (
            len(dets.shape) == 2
        ), "Unsupported 'dets' dimensions, valid number of dimensions is two"
        assert (
            dets.shape[1] == 6
        ), "Unsupported 'dets' 2nd dimension lenght, valid lenghts is 6"

        dets = np.hstack([dets, np.arange(len(dets)).reshape(-1, 1)])
        xyxy = dets[:, 0:4]
        confs = dets[:, 4]
        clss = dets[:, 5]
        det_ind = dets[:, 6]

        if len(self.tracker.tracks) >= 1:
            warp_matrix = self.cmc.apply(img, xyxy)
            for track in self.tracker.tracks:
                track.camera_update(warp_matrix)

        # extract appearance information for each detection
        if embs is not None:
            features = embs
        else:
            features = self.model.get_features(xyxy, img)
            print("feature shape:", features)


        print("Input shape of dets:", dets.shape)
        print("Input shape of img:", img.shape)
        if embs is not None:
            print("Input shape of embs:", embs.shape)


        tlwh = xyxy2tlwh(xyxy)
        detections = [
            Detection(box, conf, cls, det_ind, feat) for
            box, conf, cls, det_ind, feat in
            zip(tlwh, confs, clss, det_ind, features)
        ]

        # update tracker
        self.tracker.predict()
        self.tracker.update(detections)

        # output bbox identities
        outputs = []
        for track in self.tracker.tracks:
            if not track.is_confirmed() or track.time_since_update >= 1:
                continue

            x1, y1, x2, y2 = track.to_tlbr()

            id = track.id
            conf = track.conf
            cls = track.cls
            det_ind = track.det_ind

            outputs.append(
                np.concatenate(([x1, y1, x2, y2], [id], [conf], [cls], [det_ind])).reshape(1, -1)
            )
        if len(outputs) > 0:
            return np.concatenate(outputs)
        return np.array([])

# 2. Class ObjectDetection

## Simple use (recommend)

In [ ]:
class ObjectDetection:
    def __init__(self, model_weights="yolov8s.pt", capture_index=0, min_temporal_threshold=0, max_temporal_threshold=0, iou_threshold=0.2, use_frame_id=False):
        self.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        print("Using Device: ", self.device)
        self.model = self.load_model(model_weights)
        self.classes = self.model.names
        self.classes = model_classes
        self.colors = Colors(len(self.classes))
        self.font = cv2.FONT_HERSHEY_SIMPLEX
        self.capture_index = capture_index
        self.cap = self.load_capture()
        reid_weights = Path("/content/osnet_x0_25_endocv_30.pt")
        self.tracker = StrongSORT(reid_weights,
                                  torch.device(self.device),
                                  fp16 = False,
                                  max_dist=0.95,
                                  max_iou_dist=0.95,
                                  max_age=300
                                  )
        self.min_temporal_threshold = min_temporal_threshold
        self.max_temporal_threshold = max_temporal_threshold
        self.iou_threshold = iou_threshold
        self.use_frame_id = use_frame_id

    def load_model(self, weights):
        model = YOLO(weights)
        model.fuse()
        return model

    def predict(self, frame):
        results = self.model(frame, stream=True, verbose=False, conf=0.6, line_width=1)
        return results

    def _frame_idx_to_hmsf(self, frame_id: int):
        """convert to hmsf timestamp by given frame idx and fps"""
        self.video_fps = self.cap.get(cv2.CAP_PROP_FPS)
        assert self.video_fps
        base = datetime.strptime('00:00:00.000000', '%H:%M:%S.%f')
        delta = timedelta(seconds=frame_id/self.video_fps)
        return (base + delta).strftime('%H:%M:%S.%f')

    def _frame_idx_to_hms(self, frame_id: int):
        """convert to hms timestamp by given frame idx and fps"""
        self.video_fps = self.cap.get(cv2.CAP_PROP_FPS)
        assert self.video_fps
        base = datetime.strptime('00:00:00', '%H:%M:%S')
        delta = timedelta(seconds=frame_id//self.video_fps)
        return (base + delta).strftime('%H:%M:%S')

    def draw_tracks(self, frame, tracks, txt_file, overlap_threshold=0.5):
        seq_length = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_rate = self.cap.get(cv2.CAP_PROP_FPS)
        im_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        im_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_id = int(self.cap.get(cv2.CAP_PROP_POS_FRAMES))-1
        timestamp_hms = self._frame_idx_to_hms(frame_id)
        timestamp_hmsf = self._frame_idx_to_hmsf(frame_id)
        null_notes = "Tracking"
        for track in tracks:
            x1, y1, x2, y2 = int(track[0]), int(track[1]), int(track[2]), int(track[3])
            id = int(track[4])
            conf = round(track[5], 2)
            class_id = int(track[6])
            class_name = self.classes[class_id]
            cv2.rectangle(frame, (x1,y1), (x2, y2), self.colors(class_id), 5)
            label = f'{class_name}, ID: {id}' # hiển thị
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 5)
            cv2.rectangle(frame, (x1, y1+h+15), (x1+w, y1), self.colors(class_id), -1)
            cv2.putText(frame, label, (x1,y1+h+10), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255,255,255) , 3)
            # Ghi kết quả vào file txt
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2
            scale_height = frame.shape[0]
            scale_width = frame.shape[1]
            txt_file.write(f"{timestamp_hms},{timestamp_hmsf},{frame_id},{frame_rate},{class_name},{id},{id},{null_notes},{frame.shape[0]},{frame.shape[1]},{scale_height},{scale_width},{x1},{y1},{x2},{y2},{center_x},{center_y}\n")
            #txt_file.write(f"{int(frame_id)},{id},{x1},{y1},{x2-x1},{y2-y1},{conf},-1,-1,-1\n")

        return frame


    def load_capture(self):
        cap = cv2.VideoCapture(self.capture_index)
        assert cap.isOpened()
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        video_name = "tracking_" + input_video_name + ".mp4"
        self.writer = cv2.VideoWriter(video_name
        , cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
        return cap

    def write_seqinfo_ini(self, seq_name, seq_length, frame_rate, im_width, im_height, im_ext, im_dir):
        with open("seqinfo.ini", "w") as f:
            f.write("[Sequence]\n")
            f.write(f"name={seq_name}\n")
            f.write(f"imDir={im_dir}\n")  # Thay thế bằng thư mục chứa ảnh nếu cần
            f.write(f"frameRate={frame_rate}\n")
            f.write(f"seqLength={seq_length}\n")
            f.write(f"imWidth={im_width}\n")
            f.write(f"imHeight={im_height}\n")
            f.write(f"imExt={im_ext}\n")

    def calculate_iou(self, box1, box2):
        """
        Calculate intersection over union (IoU) between two bounding boxes.

        Parameters:
        - box1 (list): [x1, y1, x2, y2] of the first box.
        - box2 (list): [x1, y1, x2, y2] of the second box.

        Returns:
        - iou (float): Intersection over Union (IoU) value.
        """
        # Calculate intersection area
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        intersection_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1)

        # Calculate areas of each bounding box
        box1_area = (box1[2] - box1[0] + 1) * (box1[3] - box1[1] + 1)
        box2_area = (box2[2] - box2[0] + 1) * (box2[3] - box2[1] + 1)

        # Calculate union area
        union_area = box1_area + box2_area - intersection_area

        # Calculate IoU
        iou = intersection_area / union_area

        return iou

    def update_track_id(self, current_tracks, previous_tracks):
        updated_tracks = []
        for current_track in current_tracks:
            min_distance = float('inf')
            matching_track_id = None
            for previous_track in previous_tracks:
                if current_track[6] != previous_track[6]:
                    continue  # Skip tracks of different classes
                iou = self.calculate_iou(current_track[:4], previous_track[:4])
                #print(iou, self.iou_threshold)
                if iou > self.iou_threshold:
                    if self.use_frame_id:
                        time_diff = abs(current_track[3] - previous_track[3])
                        if time_diff < min_distance:
                            min_distance = time_diff
                            matching_track_id = previous_track[4]
                    else:
                        time_diff = abs(current_track[1] - previous_track[1])
                        if time_diff < min_distance:
                            min_distance = time_diff
                            matching_track_id = previous_track[4]

            if matching_track_id is not None:
                current_track[4] = matching_track_id
            updated_tracks.append(current_track)
        return updated_tracks

    def __call__(self):
        tracker = self.tracker

        # Lấy thông tin từ video kết quả
        seq_name = "StrongSort"
        im_dir = "img1"
        seq_length = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_rate = self.cap.get(cv2.CAP_PROP_FPS)
        im_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        im_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        im_ext = ".jpg"  # Phần mở rộng của ảnh

        # Ghi thông tin vào file seqinfo.ini
        self.write_seqinfo_ini(seq_name, seq_length, frame_rate, im_width, im_height, im_ext, im_dir)

        # Mở file txt để ghi kết quả
        with open("tracking_result.txt", "w") as txt_file:
            txt_file.write("timestamp_hms,timestamp_hmsf,frame_idx,fps,object_cls,object_idx,object_id,notes,frame_height,frame_width,scale_height,scale_width,x1,y1,x2,y2,center_x,center_y\n")
            previous_tracks = []
            while True:
                start_time = perf_counter()
                ret, frame = self.cap.read()
                if not ret:
                    break
                cv2.rectangle(frame, (0, 30), (220, 80), (255, 255, 255), -1)
                detections = self.predict(frame)
                for dets in detections:
                    # print("Detection:", dets.orig_img.shape)
                    # print("Detection.shape:", dets.shape)
                    det_boxes = dets.boxes.data.to("cpu").numpy()
                    # print("Shape of input tensor to ReID:", det_boxes.shape)
                    tracks = tracker.update(det_boxes, frame)
                    if len(tracks.shape) == 2 and tracks.shape[1] == 8:
                        if len(previous_tracks) > 0:
                            tracks = self.update_track_id(tracks, previous_tracks)
                        frame = self.draw_tracks(frame, tracks, txt_file)
                        previous_tracks = tracks

                end_time = perf_counter()
                fps = 1 / np.round(end_time - start_time, 2)
                cv2.putText(frame, f'FPS: {int(fps)}', (20, 70), self.font, 1.5, (0, 255, 0), 5)
                self.writer.write(frame)
                # cv2_imshow(frame)
                if cv2.waitKey(5) & 0xFF == 27:
                    break
            self.cap.release()
            self.writer.release()
            cv2.destroyAllWindows()



## Nhãn bên trái (Optional)


In [ ]:
class ObjectDetection:
    def __init__(self, model_weights="yolov8s.pt", capture_index=0, min_temporal_threshold=0, max_temporal_threshold=0, iou_threshold=0.2, use_frame_id=False):
        self.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        print("Using Device: ", self.device)
        self.model = self.load_model(model_weights)
        self.classes = self.model.names
        self.classes = model_classes
        self.colors = Colors(len(self.classes))
        self.font = cv2.FONT_HERSHEY_SIMPLEX
        self.capture_index = capture_index
        self.cap = self.load_capture()
        reid_weights = Path("/content/osnet_x0_25_endocv_30.pt")
        self.tracker = StrongSORT(reid_weights,
                                  torch.device(self.device),
                                  fp16 = False,
                                  max_dist=0.95,
                                  max_iou_dist=0.95,
                                  max_age=300
                                  )
        self.min_temporal_threshold = min_temporal_threshold
        self.max_temporal_threshold = max_temporal_threshold
        self.iou_threshold = iou_threshold
        self.use_frame_id = use_frame_id
        self.labels = {}
        self.saved_images = {}
        self.last_detected_frame = None

    def load_model(self, weights):
        model = YOLO(weights)
        model.fuse()
        return model

    def predict(self, frame):
        results = self.model(frame, stream=True, verbose=False, conf=0.6, line_width=1)
        return results

    def _frame_idx_to_hmsf(self, frame_id: int):
        """convert to hmsf timestamp by given frame idx and fps"""
        self.video_fps = self.cap.get(cv2.CAP_PROP_FPS)
        assert self.video_fps
        base = datetime.strptime('00:00:00.000000', '%H:%M:%S.%f')
        delta = timedelta(seconds=frame_id/self.video_fps)
        return (base + delta).strftime('%H:%M:%S.%f')

    def _frame_idx_to_hms(self, frame_id: int):
        """convert to hms timestamp by given frame idx and fps"""
        self.video_fps = self.cap.get(cv2.CAP_PROP_FPS)
        assert self.video_fps
        base = datetime.strptime('00:00:00', '%H:%M:%S')
        delta = timedelta(seconds=frame_id//self.video_fps)
        return (base + delta).strftime('%H:%M:%S')

    def load_capture(self):
        cap = cv2.VideoCapture(self.capture_index)
        assert cap.isOpened()
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        video_name = "strongsort_" + input_video_name + ".mp4"
        self.writer = cv2.VideoWriter(video_name
        , cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
        return cap

    def write_seqinfo_ini(self, seq_name, seq_length, frame_rate, im_width, im_height, im_ext, im_dir):
        with open("seqinfo.ini", "w") as f:
            f.write("[Sequence]\n")
            f.write(f"name={seq_name}\n")
            f.write(f"imDir={im_dir}\n")  # Thay thế bằng thư mục chứa ảnh nếu cần
            f.write(f"frameRate={frame_rate}\n")
            f.write(f"seqLength={seq_length}\n")
            f.write(f"imWidth={im_width}\n")
            f.write(f"imHeight={im_height}\n")
            f.write(f"imExt={im_ext}\n")

    def calculate_iou(self, box1, box2):
        """
        Calculate intersection over union (IoU) between two bounding boxes.

        Parameters:
        - box1 (list): [x1, y1, x2, y2] of the first box.
        - box2 (list): [x1, y1, x2, y2] of the second box.

        Returns:
        - iou (float): Intersection over Union (IoU) value.
        """
        # Calculate intersection area
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        intersection_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1)

        # Calculate areas of each bounding box
        box1_area = (box1[2] - box1[0] + 1) * (box1[3] - box1[1] + 1)
        box2_area = (box2[2] - box2[0] + 1) * (box2[3] - box2[1] + 1)

        # Calculate union area
        union_area = box1_area + box2_area - intersection_area

        # Calculate IoU
        iou = intersection_area / union_area

        return iou

    def update_track_id(self, current_tracks, previous_tracks):
        updated_tracks = []
        for current_track in current_tracks:
            min_distance = float('inf')
            matching_track_id = None
            for previous_track in previous_tracks:
                if current_track[6] != previous_track[6]:
                    continue  # Skip tracks of different classes
                iou = self.calculate_iou(current_track[:4], previous_track[:4])
                #print(iou, self.iou_threshold)
                if iou > self.iou_threshold:
                    if self.use_frame_id:
                        time_diff = abs(current_track[3] - previous_track[3])
                        if time_diff < min_distance:
                            min_distance = time_diff
                            matching_track_id = previous_track[4]
                    else:
                        time_diff = abs(current_track[1] - previous_track[1])
                        if time_diff < min_distance:
                            min_distance = time_diff
                            matching_track_id = previous_track[4]

            if matching_track_id is not None:
                current_track[4] = matching_track_id
            updated_tracks.append(current_track)
        return updated_tracks

    def draw_tracks(self, frame, tracks, txt_file, overlap_threshold=0.5):
        seq_length = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_rate = self.cap.get(cv2.CAP_PROP_FPS)
        im_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        im_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_id = int(self.cap.get(cv2.CAP_PROP_POS_FRAMES))-1
        timestamp_hms = self._frame_idx_to_hms(frame_id)
        timestamp_hmsf = self._frame_idx_to_hmsf(frame_id)
        null_notes = "Tracking"
        for track in tracks:
            x1, y1, x2, y2 = int(track[0]), int(track[1]), int(track[2]), int(track[3])
            id = int(track[4])
            conf = round(track[5], 2)
            class_id = int(track[6])
            class_name = self.classes[class_id]
            cv2.rectangle(frame, (x1,y1), (x2, y2), self.colors(class_id), 5)
            self.save_first_detected_frame(frame, track)
            # Update label if the object ID is new or changed
            if id not in self.labels:
                self.labels[id] = class_name

            # Write result to txt file
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2
            scale_height = frame.shape[0]
            scale_width = frame.shape[1]
            txt_file.write(f"{timestamp_hms},{timestamp_hmsf},{frame_id},{frame_rate},{class_name},{id},{id},{null_notes},{frame.shape[0]},{frame.shape[1]},{scale_height},{scale_width},{x1},{y1},{x2},{y2},{center_x},{center_y}\n")
            #detected_ids.add(id)
        return frame

    def display_labels(self, frame, tracks):
        # Tạo một từ điển để lưu trữ các nhãn đã được gán
        labels_dict = {}

        # Lặp qua các tracks và cập nhật từ điển labels_dict
        for track in tracks:
            id = int(track[4])
            class_id = int(track[6])
            class_name = self.classes[class_id]
            labels_dict[id] = class_name

        # Biến lưu màu của nhãn trước đó
        previous_label_colors = {}

        # Hiển thị nhãn trên khung hình
        for id, label in self.labels.items():
            label = f'{self.labels[id]}, ID: {id}'
            if id in labels_dict:
                # Nếu đối tượng có trong tracks, hiển thị nhãn mới
                self.labels[id] = labels_dict[id]
                class_id = int(track[6])
                label_color = (0, 255, 0)
                previous_label_colors[id] = label_color  # Lưu màu của nhãn mới
            else:
                # Nếu không phát hiện được đối tượng trong frame, sử dụng màu của nhãn trước đó
                label_color = previous_label_colors.get(id, (0, 0, 255))
            # Hiển thị nhãn trên khung hình
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 5)
            label_x = 0
            label_y = 50 + h
            cv2.rectangle(frame, (label_x, label_y - h - 15), (label_x + w + 10,label_y + 10), (0, 0, 0), -1)
            cv2.putText(frame, label, (label_x + 5, label_y - 5), cv2.FONT_HERSHEY_SIMPLEX, 1.5, label_color, 3)

        return frame
    def save_first_detected_frame(self, frame, track):
        x1, y1, x2, y2 = int(track[0]), int(track[1]), int(track[2]), int(track[3])
        id = int(track[4])
        class_id = int(track[6])
        key = (id, class_id)
        if key not in self.saved_images:
            object_img = frame[y1:y2, x1:x2]
            height, width = object_img.shape[:2]
            aspect_ratio = width / height
            new_width = 300
            new_height = int(new_width / aspect_ratio)
            resized_img = cv2.resize(object_img, (new_width, new_height))
            self.saved_images[key] = resized_img

    def draw_saved_images(self, frame):
        for (id, class_id), img in self.saved_images.items():
            x_offset = 20
            y_offset = 100
            y_end = y_offset + img.shape[0]
            x_end = x_offset + img.shape[1]
            frame[y_offset:y_end, x_offset:x_end] = img
        return frame


    def __call__(self):
        tracker = self.tracker

        # Lấy thông tin từ video kết quả
        seq_name = "StrongSort"
        im_dir = "img1"
        seq_length = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_rate = self.cap.get(cv2.CAP_PROP_FPS)
        im_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        im_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        im_ext = ".jpg"  # Phần mở rộng của ảnh

        # Ghi thông tin vào file seqinfo.ini
        self.write_seqinfo_ini(seq_name, seq_length, frame_rate, im_width, im_height, im_ext, im_dir)

        # Mở file txt để ghi kết quả
        with open("tracking_result.txt", "w") as txt_file:
            txt_file.write("timestamp_hms,timestamp_hmsf,frame_idx,fps,object_cls,object_idx,object_id,notes,frame_height,frame_width,scale_height,scale_width,x1,y1,x2,y2,center_x,center_y\n")
            previous_tracks = []
            while True:
                start_time = perf_counter()
                ret, frame = self.cap.read()
                if not ret:
                    break

                label = "Unknown"
                (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 5)
                label_x = 0
                label_y = 50 + h
                cv2.rectangle(frame, (label_x, label_y - h - 15), (label_x + w + 10, label_y + 10), (0, 0, 0), -1)
                cv2.putText(frame, label, (label_x + 5, label_y - 5), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)

                detections = self.predict(frame)
                for dets in detections:
                    tracks = tracker.update(dets.boxes.data.to("cpu").numpy(), frame)
                    if len(tracks.shape) == 2 and tracks.shape[1] == 8:
                        if len(previous_tracks) > 0:
                            tracks = self.update_track_id(tracks, previous_tracks)
                        frame = self.draw_tracks(frame, tracks, txt_file)
                        previous_tracks = tracks
                self.display_labels(frame, tracks)
                self.draw_saved_images(frame)
                end_time = perf_counter()
                # fps = 1 / np.round(end_time - start_time, 2)
                # cv2.rectangle(frame, (0, 30), (220, 80), (255, 255, 255), -1)
                # cv2.putText(frame, f'FPS: {int(fps)}', (20, 70), self.font, 1.5, (0, 255, 0), 5)
                self.writer.write(frame)
                # cv2_imshow(frame)
                if cv2.waitKey(5) & 0xFF == 27:
                    break
            self.cap.release()
            self.writer.release()
            cv2.destroyAllWindows()

## Nhãn bên phải (Optional)

In [ ]:
class ObjectDetection:
    def __init__(self, model_weights="yolov8s.pt", capture_index=0, min_temporal_threshold=0, max_temporal_threshold=0, iou_threshold=0.2, use_frame_id=False):
        self.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        print("Using Device: ", self.device)
        self.model = self.load_model(model_weights)
        self.classes = self.model.names
        self.classes = model_classes
        self.colors = Colors(len(self.classes))
        self.font = cv2.FONT_HERSHEY_SIMPLEX
        self.capture_index = capture_index
        self.cap = self.load_capture()
        reid_weights = Path("/content/osnet_x0_25_endocv_30.pt")
        self.tracker = StrongSORT(reid_weights,
                                  torch.device(self.device),
                                  fp16 = False,
                                  max_dist=0.95,
                                  max_iou_dist=0.95,
                                  max_age=300
                                  )
        self.min_temporal_threshold = min_temporal_threshold
        self.max_temporal_threshold = max_temporal_threshold
        self.iou_threshold = iou_threshold
        self.use_frame_id = use_frame_id
        self.labels = {}
        self.saved_images = {}
        self.last_detected_frame = None

    def load_model(self, weights):
        model = YOLO(weights)
        model.fuse()
        return model

    def predict(self, frame):
        results = self.model(frame, stream=True, verbose=False, conf=0.5, line_width=1)
        return results

    def _frame_idx_to_hmsf(self, frame_id: int):
        """convert to hmsf timestamp by given frame idx and fps"""
        self.video_fps = self.cap.get(cv2.CAP_PROP_FPS)
        assert self.video_fps
        base = datetime.strptime('00:00:00.000000', '%H:%M:%S.%f')
        delta = timedelta(seconds=frame_id/self.video_fps)
        return (base + delta).strftime('%H:%M:%S.%f')

    def _frame_idx_to_hms(self, frame_id: int):
        """convert to hms timestamp by given frame idx and fps"""
        self.video_fps = self.cap.get(cv2.CAP_PROP_FPS)
        assert self.video_fps
        base = datetime.strptime('00:00:00', '%H:%M:%S')
        delta = timedelta(seconds=frame_id//self.video_fps)
        return (base + delta).strftime('%H:%M:%S')

    def load_capture(self):
        cap = cv2.VideoCapture(self.capture_index)
        assert cap.isOpened()
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        video_name = "tracking_" + input_video_name + ".mp4"
        self.writer = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
        return cap

    def write_seqinfo_ini(self, seq_name, seq_length, frame_rate, im_width, im_height, im_ext, im_dir):
        with open("seqinfo.ini", "w") as f:
            f.write("[Sequence]\n")
            f.write(f"name={seq_name}\n")
            f.write(f"imDir={im_dir}\n")  # Thay thế bằng thư mục chứa ảnh nếu cần
            f.write(f"frameRate={frame_rate}\n")
            f.write(f"seqLength={seq_length}\n")
            f.write(f"imWidth={im_width}\n")
            f.write(f"imHeight={im_height}\n")
            f.write(f"imExt={im_ext}\n")

    def calculate_iou(self, box1, box2):
        # Calculate intersection area
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        intersection_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1)

        # Calculate areas of each bounding box
        box1_area = (box1[2] - box1[0] + 1) * (box1[3] - box1[1] + 1)
        box2_area = (box2[2] - box2[0] + 1) * (box2[3] - box2[1] + 1)

        # Calculate union area
        union_area = box1_area + box2_area - intersection_area

        # Calculate IoU
        iou = intersection_area / union_area

        return iou

    def update_track_id(self, current_tracks, previous_tracks):
        updated_tracks = []
        for current_track in current_tracks:
            min_distance = float('inf')
            matching_track_id = None
            for previous_track in previous_tracks:
                if current_track[6] != previous_track[6]:
                    continue  # Skip tracks of different classes
                iou = self.calculate_iou(current_track[:4], previous_track[:4])
                #print(iou, self.iou_threshold)
                if iou > self.iou_threshold:
                    if self.use_frame_id:
                        time_diff = abs(current_track[3] - previous_track[3])
                        if time_diff < min_distance:
                            min_distance = time_diff
                            matching_track_id = previous_track[4]
                    else:
                        time_diff = abs(current_track[1] - previous_track[1])
                        if time_diff < min_distance:
                            min_distance = time_diff
                            matching_track_id = previous_track[4]

            if matching_track_id is not None:
                current_track[4] = matching_track_id
            updated_tracks.append(current_track)
        return updated_tracks

    def draw_tracks(self, frame, tracks, txt_file, overlap_threshold=0.5):
        seq_length = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_rate = self.cap.get(cv2.CAP_PROP_FPS)
        im_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        im_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_id = int(self.cap.get(cv2.CAP_PROP_POS_FRAMES))-1
        timestamp_hms = self._frame_idx_to_hms(frame_id)
        timestamp_hmsf = self._frame_idx_to_hmsf(frame_id)
        null_notes = "Tracking"
        labels_dict = {}
        for track in tracks:
            x1, y1, x2, y2 = int(track[0]), int(track[1]), int(track[2]), int(track[3])
            id = int(track[4])
            conf = round(track[5], 2)
            class_id = int(track[6])
            class_name = self.classes[class_id]
            cv2.rectangle(frame, (x1,y1), (x2, y2), self.colors(class_id), 5)
            # self.save_first_detected_frame(frame, track)
            # Write result to txt file
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2
            scale_height = frame.shape[0]
            scale_width = frame.shape[1]
            # Update label if the object ID is new or changed
            if id not in self.labels:
                self.labels[id] = class_name
            self.save_first_detected_frame(frame, track)
            txt_file.write(f"{timestamp_hms},{timestamp_hmsf},{frame_id},{frame_rate},{class_name},{id},{id},{null_notes},{frame.shape[0]},{frame.shape[1]},{scale_height},{scale_width},{x1},{y1},{x2},{y2},{center_x},{center_y}\n")

        return frame

    def display_labels(self, frame, tracks):
        # Tạo một từ điển để lưu trữ các nhãn đã được gán
        frame_id = int(self.cap.get(cv2.CAP_PROP_POS_FRAMES))-1
        labels_dict = {}
        last_detection_times = {}
        previous_label_colors = {}

        # Lặp qua các tracks và cập nhật từ điển labels_dict
        for track in tracks:
            id = int(track[4])
            class_id = int(track[6])
            class_name = self.classes[class_id]
            labels_dict[id] = class_name
        # Hiển thị nhãn trên khung hình
        for id, label in self.labels.items():
            # label = f'{self.labels[id]}, ID: {id}'
            if id in labels_dict:
                # Nếu đối tượng có trong tracks, hiển thị nhãn mới
                self.labels[id] = labels_dict[id]
                class_id = int(track[6])
                label_color = self.colors(class_id)
                previous_label_colors[id] = label_color
                last_detection_times[id] = time()  # Lưu màu của nhãn mới
                label = f'{self.labels[id]}, ID: {id}'
            else:
                # Nếu không phát hiện được đối tượng trong frame, sử dụng màu của nhãn trước đó
                label_color = previous_label_colors.get(id, (255, 255, 255))

            self.labels = {}

            # Hiển thị nhãn trên khung hình
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 5)
            label_x = frame.shape[1] - w - 20
            label_y = 50 + h
            cv2.rectangle(frame, (label_x, label_y - h - 15), (label_x + w + 10,label_y + 10), (0, 0, 0), -1)
            cv2.putText(frame, label, (label_x + 5, label_y - 5), cv2.FONT_HERSHEY_SIMPLEX, 1.5, label_color, 3)

        return frame

    def save_first_detected_frame(self, frame, track):
        x1, y1, x2, y2 = int(track[0]), int(track[1]), int(track[2]), int(track[3])
        id = int(track[4])
        class_id = int(track[6])
        key = (id, class_id)

        if hasattr(self, 'last_saved_key') and self.last_saved_key != key:
            # Clear the saved images if there is a change in class or id
            self.saved_images.clear()

        if key not in self.saved_images:
            object_img = frame[y1:y2, x1:x2]
            height, width = object_img.shape[:2]
            #print(height, width)
            if height > 0:
              aspect_ratio = width / height
              new_width = 300
              if aspect_ratio == 0:
                new_height = 300
              else:
                new_height = int(new_width / aspect_ratio)

              if new_height > 980:
                  new_height = 980
                  new_width = int(new_height * aspect_ratio)

              resized_img = cv2.resize(object_img, (new_width, new_height))

            if height <= 0:
              resized_img = cv2.resize(object_img, (300, height))

            self.saved_images[key] = resized_img
            self.last_saved_key = key

    def draw_saved_images(self, frame):
        for (id, class_id), img in self.saved_images.items():
            x_offset = 1600
            y_offset = 100
            y_end = y_offset + img.shape[0]
            x_end = x_offset + img.shape[1]
            cv2.rectangle(frame, (1600, 100), (x_end, 1080), (0, 0, 0), -1)
            frame[y_offset:y_end, x_offset:x_end] = img
            #print(img.shape)
        return frame


    def __call__(self):
        tracker = self.tracker

        # Lấy thông tin từ video kết quả
        seq_name = "StrongSort"
        im_dir = "img1"
        seq_length = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_rate = self.cap.get(cv2.CAP_PROP_FPS)
        im_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        im_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        im_ext = ".jpg"  # Phần mở rộng của ảnh

        # Ghi thông tin vào file seqinfo.ini
        self.write_seqinfo_ini(seq_name, seq_length, frame_rate, im_width, im_height, im_ext, im_dir)

        # Mở file txt để ghi kết quả
        with open("tracking_result.txt", "w") as txt_file:
            txt_file.write("timestamp_hms,timestamp_hmsf,frame_idx,fps,object_cls,object_idx,object_id,notes,frame_height,frame_width,scale_height,scale_width,x1,y1,x2,y2,center_x,center_y\n")
            previous_tracks = []
            while True:
                start_time = perf_counter()
                ret, frame = self.cap.read()
                if not ret:
                    break

                label = "Unknown"
                (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 5)
                label_x = frame.shape[1] - w - 20
                label_y = 50 + h
                cv2.rectangle(frame, (label_x, label_y - h - 15), (label_x + w + 10, label_y + 10), (0, 0, 0), -1)
                cv2.putText(frame, label, (label_x + 5, label_y - 5), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
                detections = self.predict(frame)
                for dets in detections:
                    tracks = tracker.update(dets.boxes.data.to("cpu").numpy(), frame)
                    if len(tracks.shape) == 2 and tracks.shape[1] == 8:
                        if len(previous_tracks) > 0:
                            tracks = self.update_track_id(tracks, previous_tracks)
                        frame = self.draw_tracks(frame, tracks, txt_file)
                        previous_tracks = tracks
                self.display_labels(frame, tracks)
                self.draw_saved_images(frame)
                end_time = perf_counter()
                # fps = 1 / np.round(end_time - start_time, 2)
                # cv2.rectangle(frame, (0, 30), (220, 80), (255, 255, 255), -1)
                # cv2.putText(frame, f'FPS: {int(fps)}', (20, 70), self.font, 1.5, (0, 255, 0), 5)
                self.writer.write(frame)
                #cv2_imshow(frame)
                if cv2.waitKey(5) & 0xFF == 27:
                    break
            self.cap.release()
            self.writer.release()
            cv2.destroyAllWindows()

# 3. RUN

In [ ]:
detector = ObjectDetection(model_weights, test_vid)
detector()
video_name = "tracking_" + input_video_name + ".mp4"
print(video_name)

Using Device:  cuda:0
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs


2025-02-26 13:58:33.450 | SUCCESS  | boxmot.appearance.reid_model_factory:load_pretrained_weights:207 - Successfully loaded pretrained weights from "/content/osnet_x0_25_endocv_30.pt"


feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: []
Input shape of dets: (0, 7)
Input shape of img: (1080, 1920, 3)
feature shape: [

KeyboardInterrupt: 

In [ ]:
# Giả sử bạn đã có một instance của StrongSORT
strongsort = StrongSORT(model_weights="osnet_x0_25_msmt17.pt", device="cuda:0", fp16=False)

# Dữ liệu đầu vào
dets = np.array([
    [100, 50, 200, 150, 0.9, 1],  # (x1, y1, x2, y2, confidence, class)
    [300, 100, 400, 200, 0.8, 0]
])
img = np.random.randint(0, 255, (640, 480, 3), dtype=np.uint8)  # Ảnh RGB giả lập

# Tính embs từ bounding boxes
xyxy = dets[:, :4]  # Lấy tọa độ bounding boxes
embs = strongsort.model.get_features(xyxy, img)
print("embs:", embs.shape)
# Cập nhật tracker với embeddings
outputs = strongsort.update(dets, img, embs)

print("Tracking Output:", outputs)


AttributeError: 'str' object has no attribute 'type'

# 4. Generate txt csv results

In [ ]:
def txt_to_csv(input_txt_file, output_csv_file):
    with open(input_txt_file, 'r') as infile, open(output_csv_file, 'w', newline='') as outfile:
        reader = csv.reader(infile, delimiter=',')
        writer = csv.writer(outfile)

        for row in reader:
            writer.writerow(row)


def convert_file(input_file, output_file):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        lines = infile.readlines()
        for line in lines[1:]:  # Skip the first line (header)
            parts = line.strip().split(',')
            if len(parts) < 17:
                continue  # Skip lines that do not have enough values

            frame_id = parts[2]
            object_id = parts[5]
            x1 = int(parts[12])
            y1 = int(parts[13])
            x2 = int(parts[14])
            y2 = int(parts[15])
            conf = round(float(parts[6]), 2)

            width = x2 - x1
            height = y2 - y1

            # Write to the output file
            outfile.write(f"{frame_id},{object_id},{x1},{y1},{width},{height},{conf},-1,-1,-1\n")

# Usage
input_file = 'tracking_result.txt'
output_mot_file = 'mot_result.txt'
output_csv_file = "tracking_" + input_video_name +'.csv'
convert_file(input_file, output_mot_file)
txt_to_csv(input_file, output_csv_file)

In [ ]:
# Tạo thư mục với tên giống với video_name trong /content/run
run_folder = "/content/runs"
video_name = "tracking_" + input_video_name + ".mp4"
video_folder = os.path.join(run_folder, video_name)
if not os.path.exists(video_folder):
    os.makedirs(video_folder)

# Di chuyển video, seqinfo.ini và results.txt vào thư mục vừa tạo
os.rename(video_name, os.path.join(video_folder, video_name))
os.rename("seqinfo.ini", os.path.join(video_folder, "seqinfo.ini"))
os.rename("mot_result.txt", os.path.join(video_folder, "mot_result.txt"))
os.rename("tracking_result.txt", os.path.join(video_folder, "tracking_result.txt"))
os.rename(output_csv_file, os.path.join(video_folder, output_csv_file))
#os.rename('detect_'+input_video_name+'.mp4', os.path.join(video_folder,'detect_'+input_video_name+'.mp4'))
print(video_name)

FileNotFoundError: [Errno 2] No such file or directory: 'tracking_UTTQ_230411BVK107.mp4' -> '/content/runs/tracking_UTTQ_230411BVK107.mp4/tracking_UTTQ_230411BVK107.mp4'

In [ ]:
from google.colab import drive
import shutil
import os
from google.colab import files



# Step 2: Define the folder to be zipped and the output zip file path
folder_path = '/content/runs'
output_zip_path = '/content/run.zip'

# Step 3: Zip the folder
shutil.make_archive(output_zip_path.replace('.zip', ''), 'zip', folder_path)

# Step 4: Verify the zip file is created
if os.path.exists(output_zip_path):
    print(f'Zip file created successfully: {output_zip_path}')
else:
    print('Error in creating zip file')

# Step 5: Download the zip file
#files.download(output_zip_path)


Zip file created successfully: /content/run2006.zip


In [ ]:
!cp -r /content/run.zip /content/drive/MyDrive

In [ ]:
from __future__ import division, absolute_import
import torch
from torch import nn
from torch.nn import functional as F

__all__ = ['osnet_avgpool', 'osnet_maxpool']


##########
# SE Block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool3d(1)  # GAP để nén feature map thành 1 vector
        self.fc1 = nn.Linear(in_channels, in_channels // reduction)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch, channels, depth, height, width = x.size()
        y = self.global_avg_pool(x).view(batch, channels)  # [B, C, 1, 1, 1] -> [B, C]
        y = self.fc1(y)
        y = F.relu(y)
        y = self.fc2(y)
        y = self.sigmoid(y).view(batch, channels, 1, 1, 1)  # Scale lại kích thước
        return x * y  # Nhân trọng số với feature map gốc


##########
#  CBAM Module (Channel & Spatial Attention)
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.max_pool = nn.AdaptiveMaxPool3d(1)

        self.fc1 = nn.Conv3d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv3d(in_planes // ratio, in_planes, 1, bias=False)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv3d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv1(out)
        return self.sigmoid(out)

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

class C3D_CBAM(nn.Module):
    def __init__(self):
        super(C3D_CBAM, self).__init__()
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.cbam1 = CBAM(64)
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.cbam2 = CBAM(128)
        self.pool2 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.cbam3a = CBAM(256)
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.cbam3b = CBAM(256)
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.cbam4a = CBAM(512)
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.cbam4b = CBAM(512)
        self.pool4 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))

        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.cbam5a = CBAM(512)
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.cbam5b = CBAM(512)
        # self.pool5 = nn.MaxPool3d(kernel_size=2, stride=2)
        self.pool5 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=(0, 1, 1))


    def forward(self, x):
        print("Input shape:", x.shape)
        x = self.pool1(self.cbam1(F.relu(self.conv1(x))))
        print("After pool1:", x.shape)
        x = self.pool2(self.cbam2(F.relu(self.conv2(x))))
        print("After pool2:", x.shape)
        x = self.pool3(self.cbam3b(F.relu(self.conv3b(self.cbam3a(F.relu(self.conv3a(x)))))))
        print("After pool3:", x.shape)
        x = self.pool4(self.cbam4b(F.relu(self.conv4b(self.cbam4a(F.relu(self.conv4a(x)))))))
        print("After pool4:", x.shape)
        x = self.pool5(self.cbam5b(F.relu(self.conv5b(self.cbam5a(F.relu(self.conv5a(x)))))))
        print("After pool5:", x.shape)
        x = x.view(x.size(0), x.size(1), -1)
        # x = x.view(x.size(0), -1)
        # Giữ lại chiều không gian cuối
        print("After flatten:", x.shape)
        return x


##########
# Basic layers
##########
class ConvLayer(nn.Module):
    """Convolution layer."""

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        groups=1
    ):
        super(ConvLayer, self).__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
            groups=groups
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class Conv1x1(nn.Module):
    """1x1 convolution."""

    def __init__(self, in_channels, out_channels, stride=1, groups=1):
        super(Conv1x1, self).__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            1,
            stride=stride,
            padding=0,
            bias=False,
            groups=groups
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class Conv1x1Linear(nn.Module):
    """1x1 convolution without non-linearity."""

    def __init__(self, in_channels, out_channels, stride=1):
        super(Conv1x1Linear, self).__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, 1, stride=stride, padding=0, bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x


class Conv3x3(nn.Module):
    """3x3 convolution."""

    def __init__(self, in_channels, out_channels, stride=1, groups=1):
        super(Conv3x3, self).__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            3,
            stride=stride,
            padding=1,
            bias=False,
            groups=groups
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class LightConv3x3(nn.Module):
    """Lightweight 3x3 convolution.

    1x1 (linear) + dw 3x3 (nonlinear).
    """

    def __init__(self, in_channels, out_channels):
        super(LightConv3x3, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 1, stride=1, padding=0, bias=False
        )
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            3,
            stride=1,
            padding=1,
            bias=False,
            groups=out_channels
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


##########
# Building blocks for omni-scale feature learning
##########
class ChannelGate(nn.Module):
    """A mini-network that generates channel-wise gates conditioned on input."""

    def __init__(
        self,
        in_channels,
        num_gates=None,
        return_gates=False,
        gate_activation='sigmoid',
        reduction=16,
        layer_norm=False
    ):
        super(ChannelGate, self).__init__()
        if num_gates is None:
            num_gates = in_channels
        self.return_gates = return_gates
        self.global_avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(
            in_channels,
            in_channels // reduction,
            kernel_size=1,
            bias=True,
            padding=0
        )
        self.norm1 = None
        if layer_norm:
            self.norm1 = nn.LayerNorm((in_channels // reduction, 1, 1))
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(
            in_channels // reduction,
            num_gates,
            kernel_size=1,
            bias=True,
            padding=0
        )
        if gate_activation == 'sigmoid':
            self.gate_activation = nn.Sigmoid()
        elif gate_activation == 'relu':
            self.gate_activation = nn.ReLU(inplace=True)
        elif gate_activation == 'linear':
            self.gate_activation = None
        else:
            raise RuntimeError(
                "Unknown gate activation: {}".format(gate_activation)
            )

    def forward(self, x):
        input = x
        x = self.global_avgpool(x)
        x = self.fc1(x)
        if self.norm1 is not None:
            x = self.norm1(x)
        x = self.relu(x)
        x = self.fc2(x)
        if self.gate_activation is not None:
            x = self.gate_activation(x)
        if self.return_gates:
            return x
        return input * x


class OSBlock(nn.Module):
    """Omni-scale feature learning block."""

    def __init__(self, in_channels, out_channels, **kwargs):
        super(OSBlock, self).__init__()
        mid_channels = out_channels // 4
        self.conv1 = Conv1x1(in_channels, mid_channels)
        self.conv2a = LightConv3x3(mid_channels, mid_channels)
        self.conv2b = nn.Sequential(
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
        )
        self.conv2c = nn.Sequential(
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
        )
        self.conv2d = nn.Sequential(
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
        )
        self.gate = ChannelGate(mid_channels)
        self.conv3 = Conv1x1Linear(mid_channels, out_channels)
        self.downsample = None
        if in_channels != out_channels:
            self.downsample = Conv1x1Linear(in_channels, out_channels)

    def forward(self, x):
        residual = x
        x1 = self.conv1(x)
        x2a = self.conv2a(x1)
        x2b = self.conv2b(x1)
        x2c = self.conv2c(x1)
        x2d = self.conv2d(x1)
        x2 = self.gate(x2a) + self.gate(x2b) + self.gate(x2c) + self.gate(x2d)
        x3 = self.conv3(x2)
        if self.downsample is not None:
            residual = self.downsample(residual)
        out = x3 + residual
        return F.relu(out)


##########
# Network architecture
##########
class BaseNet(nn.Module):

    def _make_layer(
        self, block, layer, in_channels, out_channels, reduce_spatial_size
    ):
        layers = []

        layers.append(block(in_channels, out_channels))
        for i in range(1, layer):
            layers.append(block(out_channels, out_channels))

        if reduce_spatial_size:
            layers.append(
                nn.Sequential(
                    Conv1x1(out_channels, out_channels),
                    nn.AvgPool2d(2, stride=2)
                )
            )

        return nn.Sequential(*layers)

    def _construct_fc_layer(self, fc_dims, input_dim, dropout_p=None):
        if fc_dims is None or fc_dims < 0:
            self.feature_dim = input_dim
            return None

        if isinstance(fc_dims, int):
            fc_dims = [fc_dims]

        layers = []
        for dim in fc_dims:
            layers.append(nn.Linear(input_dim, dim))
            layers.append(nn.BatchNorm1d(dim))
            layers.append(nn.ReLU(inplace=True))
            if dropout_p is not None:
                layers.append(nn.Dropout(p=dropout_p))
            input_dim = dim

        self.feature_dim = fc_dims[-1]

        return nn.Sequential(*layers)

    def init_params(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_out', nonlinearity='relu'
                )
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)


class OSNet(BaseNet):

    def __init__(
        self,
        num_classes,
        blocks,
        layers,
        channels,
        feature_dim=512,
        loss='softmax',
        pool='avg',
        **kwargs
    ):
        super(OSNet, self).__init__()
        num_blocks = len(blocks)
        assert num_blocks == len(layers)
        assert num_blocks == len(channels) - 1
        self.loss = loss

        # convolutional backbone
        self.conv1 = ConvLayer(3, channels[0], 7, stride=2, padding=3)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.conv2 = self._make_layer(
            blocks[0],
            layers[0],
            channels[0],
            channels[1],
            reduce_spatial_size=True
        )
        self.conv3 = self._make_layer(
            blocks[1],
            layers[1],
            channels[1],
            channels[2],
            reduce_spatial_size=True
        )
        self.conv4 = self._make_layer(
            blocks[2],
            layers[2],
            channels[2],
            channels[3],
            reduce_spatial_size=False
        )
        self.conv5 = Conv1x1(channels[3], channels[3])
        if pool == 'avg':
            self.global_pool = nn.AdaptiveAvgPool2d(1)
        else:
            self.global_pool = nn.AdaptiveMaxPool2d(1)
        # fully connected layer
        self.fc = self._construct_fc_layer(
            feature_dim, channels[3], dropout_p=None
        )
        # classification layer
        self.classifier = nn.Linear(self.feature_dim, num_classes)

        self.init_params()

    def featuremaps(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        return x

    def forward(self, x):
        x = self.featuremaps(x)
        v = self.global_pool(x)
        v = v.view(v.size(0), -1)
        if self.fc is not None:
            v = self.fc(v)
        y = self.classifier(v)
        if not self.training:
            y = torch.sigmoid(y)
        return y


def osnet_avgpool(num_classes=1000, loss='softmax', **kwargs):
    return OSNet(
        num_classes,
        blocks=[OSBlock, OSBlock, OSBlock],
        layers=[2, 2, 2],
        channels=[64, 256, 384, 512],
        loss=loss,
        pool='avg',
        **kwargs
    )


def osnet_maxpool(num_classes=1000, loss='softmax', **kwargs):
    return OSNet(
        num_classes,
        blocks=[OSBlock, OSBlock, OSBlock],
        layers=[2, 2, 2],
        channels=[64, 256, 384, 512],
        loss=loss,
        pool='max',
        **kwargs
    )



class OSNet_C3D_CBAM(nn.Module):
    def __init__(self, num_classes, feature_dim=512):
        super(OSNet_C3D_CBAM, self).__init__()
        self.c3d_cbam = C3D_CBAM()
        self.osnet = OSNet(
            num_classes,
            blocks=[OSBlock, OSBlock, OSBlock],
            layers=[2, 2, 2],
            channels=[64, 256, 384, 512],
            feature_dim=512,
            pool='avg'
        )
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(512*50, feature_dim)
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        x = self.c3d_cbam(x)
        x = self.global_pool(x)
        # x = torch.flatten(x, 1)
        # x = self.fc(x)
        x = torch.flatten(x, start_dim=1, end_dim=-1)  # Chuyển từ [1, 512, 50] -> [1, 512*50]
        # x = self.fc(x)  # lỗi ở đây
        # x = self.classifier(x)
        # x = x.squeeze(2)  # Remove the temporal dimension after C3D
        x = self.osnet(x)
        return x

def example_usage():
    model = OSNet_C3D_CBAM(num_classes=10, feature_dim=512)
    input_tensor = torch.randn(1, 3, 16, 128, 128)  # (batch_size, channels, depth, height, width)
    output = model(input_tensor)
    print(output)
    print("Output shape:", output.shape)

example_usage()


Input shape: torch.Size([1, 3, 16, 128, 128])
After pool1: torch.Size([1, 64, 16, 64, 64])
After pool2: torch.Size([1, 128, 16, 32, 32])
After pool3: torch.Size([1, 256, 8, 16, 16])
After pool4: torch.Size([1, 512, 4, 8, 8])
After pool5: torch.Size([1, 512, 2, 5, 5])
After flatten: torch.Size([1, 512, 50])


RuntimeError: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [1, 1]

In [2]:
from __future__ import division, absolute_import
import warnings
import torch
from torch import nn
from torch.nn import functional as F

__all__ = [
    'osnet_x1_0', 'osnet_x0_75', 'osnet_x0_5', 'osnet_x0_25', 'osnet_ibn_x1_0', 'osnet_x0_25_endocv'
]

pretrained_urls = {
    'osnet_x1_0':
    'https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY',
    'osnet_x0_75':
    'https://drive.google.com/uc?id=1uwA9fElHOk3ZogwbeY5GkLI6QPTX70Hq',
    'osnet_x0_5':
    'https://drive.google.com/uc?id=16DGLbZukvVYgINws8u8deSaOqjybZ83i',
    'osnet_x0_25':
    'https://drive.google.com/uc?id=1rb8UN5ZzPKRc_xvtHlyDh-cSz88YX9hs',
    'osnet_ibn_x1_0':
    'https://drive.google.com/uc?id=1sr90V6irlYYDd4_4ISU2iruoRG8J__6l',
    'osnet_x0_25_endocv':
    'https://drive.google.com/uc?id=1W8mz6skAUmg33zMVpWn6woJym7xGygjh'
}


##########
# Basic layers
##########
class ConvLayer(nn.Module):
    """Convolution layer (conv + bn + relu)."""

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        groups=1,
        IN=False
    ):
        super(ConvLayer, self).__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
            groups=groups
        )
        if IN:
            self.bn = nn.InstanceNorm2d(out_channels, affine=True)
        else:
            self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class Conv1x1(nn.Module):
    """1x1 convolution + bn + relu."""

    def __init__(self, in_channels, out_channels, stride=1, groups=1):
        super(Conv1x1, self).__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            1,
            stride=stride,
            padding=0,
            bias=False,
            groups=groups
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class Conv1x1Linear(nn.Module):
    """1x1 convolution + bn (w/o non-linearity)."""

    def __init__(self, in_channels, out_channels, stride=1):
        super(Conv1x1Linear, self).__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, 1, stride=stride, padding=0, bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x


class Conv3x3(nn.Module):
    """3x3 convolution + bn + relu."""

    def __init__(self, in_channels, out_channels, stride=1, groups=1):
        super(Conv3x3, self).__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            3,
            stride=stride,
            padding=1,
            bias=False,
            groups=groups
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class LightConv3x3(nn.Module):
    """Lightweight 3x3 convolution.

    1x1 (linear) + dw 3x3 (nonlinear).
    """

    def __init__(self, in_channels, out_channels):
        super(LightConv3x3, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 1, stride=1, padding=0, bias=False
        )
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            3,
            stride=1,
            padding=1,
            bias=False,
            groups=out_channels
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


##########
# Building blocks for omni-scale feature learning
##########
class ChannelGate(nn.Module):
    """A mini-network that generates channel-wise gates conditioned on input tensor."""

    def __init__(
        self,
        in_channels,
        num_gates=None,
        return_gates=False,
        gate_activation='sigmoid',
        reduction=16,
        layer_norm=False
    ):
        super(ChannelGate, self).__init__()
        if num_gates is None:
            num_gates = in_channels
        self.return_gates = return_gates
        self.global_avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(
            in_channels,
            in_channels // reduction,
            kernel_size=1,
            bias=True,
            padding=0
        )
        self.norm1 = None
        if layer_norm:
            self.norm1 = nn.LayerNorm((in_channels // reduction, 1, 1))
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(
            in_channels // reduction,
            num_gates,
            kernel_size=1,
            bias=True,
            padding=0
        )
        if gate_activation == 'sigmoid':
            self.gate_activation = nn.Sigmoid()
        elif gate_activation == 'relu':
            self.gate_activation = nn.ReLU(inplace=True)
        elif gate_activation == 'linear':
            self.gate_activation = None
        else:
            raise RuntimeError(
                "Unknown gate activation: {}".format(gate_activation)
            )

    def forward(self, x):
        input = x
        x = self.global_avgpool(x)
        x = self.fc1(x)
        if self.norm1 is not None:
            x = self.norm1(x)
        x = self.relu(x)
        x = self.fc2(x)
        if self.gate_activation is not None:
            x = self.gate_activation(x)
        if self.return_gates:
            return x
        return input * x


class OSBlock(nn.Module):
    """Omni-scale feature learning block."""

    def __init__(
        self,
        in_channels,
        out_channels,
        IN=False,
        bottleneck_reduction=4,
        **kwargs
    ):
        super(OSBlock, self).__init__()
        mid_channels = out_channels // bottleneck_reduction
        self.conv1 = Conv1x1(in_channels, mid_channels)
        self.conv2a = LightConv3x3(mid_channels, mid_channels)
        self.conv2b = nn.Sequential(
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
        )
        self.conv2c = nn.Sequential(
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
        )
        self.conv2d = nn.Sequential(
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
            LightConv3x3(mid_channels, mid_channels),
        )
        self.gate = ChannelGate(mid_channels)
        self.conv3 = Conv1x1Linear(mid_channels, out_channels)
        self.downsample = None
        if in_channels != out_channels:
            self.downsample = Conv1x1Linear(in_channels, out_channels)
        self.IN = None
        if IN:
            self.IN = nn.InstanceNorm2d(out_channels, affine=True)

    def forward(self, x):
        identity = x
        x1 = self.conv1(x)
        x2a = self.conv2a(x1)
        x2b = self.conv2b(x1)
        x2c = self.conv2c(x1)
        x2d = self.conv2d(x1)
        x2 = self.gate(x2a) + self.gate(x2b) + self.gate(x2c) + self.gate(x2d)
        x3 = self.conv3(x2)
        if self.downsample is not None:
            identity = self.downsample(identity)
        out = x3 + identity
        if self.IN is not None:
            out = self.IN(out)
        return F.relu(out)


##########
# Network architecture
##########
class OSNet(nn.Module):
    """Omni-Scale Network.

    Reference:
        - Zhou et al. Omni-Scale Feature Learning for Person Re-Identification. ICCV, 2019.
        - Zhou et al. Learning Generalisable Omni-Scale Representations
          for Person Re-Identification. TPAMI, 2021.
    """

    def __init__(
        self,
        num_classes,
        blocks,
        layers,
        channels,
        feature_dim=512,
        loss='softmax',
        IN=False,
        **kwargs
    ):
        super(OSNet, self).__init__()
        num_blocks = len(blocks)
        assert num_blocks == len(layers)
        assert num_blocks == len(channels) - 1
        self.loss = loss
        self.feature_dim = feature_dim

        # convolutional backbone
        self.conv1 = ConvLayer(3, channels[0], 7, stride=2, padding=3, IN=IN)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.conv2 = self._make_layer(
            blocks[0],
            layers[0],
            channels[0],
            channels[1],
            reduce_spatial_size=True,
            IN=IN
        )
        self.conv3 = self._make_layer(
            blocks[1],
            layers[1],
            channels[1],
            channels[2],
            reduce_spatial_size=True
        )
        self.conv4 = self._make_layer(
            blocks[2],
            layers[2],
            channels[2],
            channels[3],
            reduce_spatial_size=False
        )
        self.conv5 = Conv1x1(channels[3], channels[3])
        self.global_avgpool = nn.AdaptiveAvgPool2d(1)
        # fully connected layer
        self.fc = self._construct_fc_layer(
            self.feature_dim, channels[3], dropout_p=None
        )
        # identity classification layer
        self.classifier = nn.Linear(self.feature_dim, num_classes)

        self._init_params()

    def _make_layer(
        self,
        block,
        layer,
        in_channels,
        out_channels,
        reduce_spatial_size,
        IN=False
    ):
        layers = []

        layers.append(block(in_channels, out_channels, IN=IN))
        for i in range(1, layer):
            layers.append(block(out_channels, out_channels, IN=IN))

        if reduce_spatial_size:
            layers.append(
                nn.Sequential(
                    Conv1x1(out_channels, out_channels),
                    nn.AvgPool2d(2, stride=2)
                )
            )

        return nn.Sequential(*layers)

    def _construct_fc_layer(self, fc_dims, input_dim, dropout_p=None):
        if fc_dims is None or fc_dims < 0:
            self.feature_dim = input_dim
            return None

        if isinstance(fc_dims, int):
            fc_dims = [fc_dims]

        layers = []
        for dim in fc_dims:
            layers.append(nn.Linear(input_dim, dim))
            layers.append(nn.BatchNorm1d(dim))
            layers.append(nn.ReLU(inplace=True))
            if dropout_p is not None:
                layers.append(nn.Dropout(p=dropout_p))
            input_dim = dim

        self.feature_dim = fc_dims[-1]

        return nn.Sequential(*layers)

    def _init_params(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_out', nonlinearity='relu'
                )
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def featuremaps(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        return x

    def forward(self, x, return_featuremaps=False):
        x = self.featuremaps(x)
        if return_featuremaps:
            return x
        v = self.global_avgpool(x)
        v = v.view(v.size(0), -1)
        if self.fc is not None:
            v = self.fc(v)
        if not self.training:
            return v
        y = self.classifier(v)
        if self.loss == 'softmax':
            return y
        elif self.loss == 'triplet':
            return y, v
        else:
            raise KeyError("Unsupported loss: {}".format(self.loss))


def init_pretrained_weights(model, key=''):
    """Initializes model with pretrained weights.

    Layers that don't match with pretrained layers in name or size are kept unchanged.
    """
    import os
    import errno
    import gdown
    from collections import OrderedDict

    def _get_torch_home():
        ENV_TORCH_HOME = 'TORCH_HOME'
        ENV_XDG_CACHE_HOME = 'XDG_CACHE_HOME'
        DEFAULT_CACHE_DIR = '~/.cache'
        torch_home = os.path.expanduser(
            os.getenv(
                ENV_TORCH_HOME,
                os.path.join(
                    os.getenv(ENV_XDG_CACHE_HOME, DEFAULT_CACHE_DIR), 'torch'
                )
            )
        )
        return torch_home

    torch_home = _get_torch_home()
    model_dir = os.path.join(torch_home, 'checkpoints')
    try:
        os.makedirs(model_dir)
    except OSError as e:
        if e.errno == errno.EEXIST:
            # Directory already exists, ignore.
            pass
        else:
            # Unexpected OSError, re-raise.
            raise
    filename = key + '_imagenet.pth'
    cached_file = os.path.join(model_dir, filename)

    if not os.path.exists(cached_file):
      if key == 'osnet_x0_25_endocv':
        filename = key + '.pth'
        cached_file = os.path.join(model_dir, filename)
        gdown.download(pretrained_urls[key], cached_file, quiet=False)
      else:
        gdown.download(pretrained_urls[key], cached_file, quiet=False)

    state_dict = torch.load(cached_file)
    model_dict = model.state_dict()
    new_state_dict = OrderedDict()
    matched_layers, discarded_layers = [], []

    for k, v in state_dict.items():
        if k.startswith('module.'):
            k = k[7:] # discard module.

        if k in model_dict and model_dict[k].size() == v.size():
            new_state_dict[k] = v
            matched_layers.append(k)
        else:
            discarded_layers.append(k)

    model_dict.update(new_state_dict)
    model.load_state_dict(model_dict)

    if len(matched_layers) == 0:
        warnings.warn(
            'The pretrained weights from "{}" cannot be loaded, '
            'please check the key names manually '
            '(** ignored and continue **)'.format(cached_file)
        )
    else:
        print(
            'Successfully loaded imagenet pretrained weights from "{}"'.
            format(cached_file)
        )
        if len(discarded_layers) > 0:
            print(
                '** The following layers are discarded '
                'due to unmatched keys or layer size: {}'.
                format(discarded_layers)
            )


##########
# Instantiation
##########
def osnet_x1_0(num_classes=1000, pretrained=True, loss='softmax', **kwargs):
    # standard size (width x1.0)
    model = OSNet(
        num_classes,
        blocks=[OSBlock, OSBlock, OSBlock],
        layers=[2, 2, 2],
        channels=[64, 256, 384, 512],
        loss=loss,
        **kwargs
    )
    if pretrained:
        init_pretrained_weights(model, key='osnet_x1_0')
    return model


def osnet_x0_75(num_classes=1000, pretrained=True, loss='softmax', **kwargs):
    # medium size (width x0.75)
    model = OSNet(
        num_classes,
        blocks=[OSBlock, OSBlock, OSBlock],
        layers=[2, 2, 2],
        channels=[48, 192, 288, 384],
        loss=loss,
        **kwargs
    )
    if pretrained:
        init_pretrained_weights(model, key='osnet_x0_75')
    return model


def osnet_x0_5(num_classes=1000, pretrained=True, loss='softmax', **kwargs):
    # tiny size (width x0.5)
    model = OSNet(
        num_classes,
        blocks=[OSBlock, OSBlock, OSBlock],
        layers=[2, 2, 2],
        channels=[32, 128, 192, 256],
        loss=loss,
        **kwargs
    )
    if pretrained:
        init_pretrained_weights(model, key='osnet_x0_5')
    return model


def osnet_x0_25(num_classes=1000, pretrained=True, loss='softmax', **kwargs):
    # very tiny size (width x0.25)
    model = OSNet(
        num_classes,
        blocks=[OSBlock, OSBlock, OSBlock],
        layers=[2, 2, 2],
        channels=[16, 64, 96, 128],
        loss=loss,
        **kwargs
    )
    if pretrained:
        init_pretrained_weights(model, key='osnet_x0_25')
    return model


def osnet_x0_25_endocv(num_classes=1000, pretrained=True, loss='softmax', **kwargs):
    # very tiny size (width x0.25)
    model = OSNet(
        num_classes,
        blocks=[OSBlock, OSBlock, OSBlock],
        layers=[2, 2, 2],
        channels=[16, 64, 96, 128],
        loss=loss,
        **kwargs
    )
    if pretrained:
        init_pretrained_weights(model, key='osnet_x0_25_endocv')
    return model


def osnet_ibn_x1_0(
    num_classes=1000, pretrained=True, loss='softmax', **kwargs
):
    # standard size (width x1.0) + IBN layer
    # Ref: Pan et al. Two at Once: Enhancing Learning and Generalization Capacities via IBN-Net. ECCV, 2018.
    model = OSNet(
        num_classes,
        blocks=[OSBlock, OSBlock, OSBlock],
        layers=[2, 2, 2],
        channels=[64, 256, 384, 512],
        loss=loss,
        IN=True,
        **kwargs
    )
    if pretrained:
        init_pretrained_weights(model, key='osnet_ibn_x1_0')
    return model

In [10]:
import torch
import torch.nn as nn
import torchvision.models as models

# -----------------------------
# 1. Thay thế Stem Layer bằng C3D
# -----------------------------
# class C3DStem(nn.Module):
#     def __init__(self):
#         super(C3DStem, self).__init__()
#         self.conv1a = nn.Conv3d(3, 64, kernel_size=3, padding=1)
#         self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)

#     def forward(self, x):
#         x = x.unsqueeze(2)  # Thêm chiều thời gian giả lập
#         x = self.conv1a(x)
#         x = self.pool1(x)
#         x = x.squeeze(2)  # Loại bỏ chiều thời gian
#         return x


class OSNetC3D_Stem(nn.Module):
    def __init__(self, osnet_model):
        super(OSNetC3D_Stem, self).__init__()
        self.stem = C3DStem()
        self.osnet = osnet_model

    def forward(self, x):
        x = self.stem(x)
        x = self.osnet(x)
        return x

# -----------------------------
# 2. Chèn C3D giữa OSBlocks
# -----------------------------
class C3D_Block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(C3D_Block, self).__init__()
        self.conv3d = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1)
        self.pool3d = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        x = x.unsqueeze(2)
        x = self.conv3d(x)
        x = self.pool3d(x)
        x = x.squeeze(2)
        return x

class OSNetC3D_Middle(nn.Module):
    def __init__(self, osnet_model):
        super(OSNetC3D_Middle, self).__init__()
        self.osnet = osnet_model
        self.c3d1 = C3D_Block(64, 64)
        self.c3d2 = C3D_Block(128, 128)

    def forward(self, x):
        x = self.osnet.conv1(x)
        x = self.osnet.maxpool(x)
        x = self.c3d1(x)
        x = self.osnet.layer1(x)
        x = self.c3d2(x)
        x = self.osnet.layer2(x)
        x = self.osnet.layer3(x)
        x = self.osnet.global_avgpool(x)
        return x

# -----------------------------
# 3. Chạy C3D song song với OSNet và ghép đặc trưng
# -----------------------------
class OSNetC3D_Parallel(nn.Module):
    def __init__(self, osnet_model, c3d_model):
        super(OSNetC3D_Parallel, self).__init__()
        self.osnet = osnet_model
        self.c3d = c3d_model
        self.fc = nn.Linear(4096 + 512, 512)  # Kết hợp đặc trưng từ OSNet và C3D

    # def forward(self, x):
    #     feat_osnet = self.osnet(x)
    #     feat_c3d = self.c3d(x.unsqueeze(2)).squeeze(2)
    #     feat_combined = torch.cat((feat_osnet, feat_c3d), dim=1)
    #     out = self.fc(feat_combined)
    #     return out

    def forward(self, x):
        b, t, c, h, w = x.shape
        if t < 3:
            x = F.interpolate(x, size=(3, h, w), mode="trilinear", align_corners=False)  # Đảm bảo có ít nhất 3 frame

        x3d = x.permute(0, 2, 1, 3, 4)  # Định dạng phù hợp cho Conv3D
        x3d = self.c3d_stem(x3d)
        x3d = torch.flatten(x3d, start_dim=1)

        x2d = x[:, t // 2, :, :, :]
        x2d = self.osnet(x2d)

        x = torch.cat([x3d, x2d], dim=1)
        return x


# -----------------------------
# Khởi tạo mô hình và kiểm tra
# -----------------------------
osnet_model = osnet_x0_25()
c3d_model = C3DStem()

model1 = OSNetC3D_Stem(osnet_model)
model2 = OSNetC3D_Middle(osnet_model)
model3 = OSNetC3D_Parallel(osnet_model, c3d_model)

x = torch.randn(1, 3, 256, 128)  # Dữ liệu đầu vào (batch, channel, height, width)

# out1 = model1(x)
out2 = model2(x)
out3 = model3(x)

print("Output shapes:")
print("Model 1 (C3D Stem):", out1.shape)
print("Model 2 (C3D Middle):", out2.shape)
print("Model 3 (C3D Parallel):", out3.shape)


RuntimeError: Given input size: (64x1x256x128). Calculated output size: (64x0x128x64). Output size is too small

In [15]:
import cv2

# Đường dẫn đến video gốc
input_video_path = '/content/video_test/UTTQ/230411BVK107.mp4'
# Đường dẫn để lưu video mới
output_video_path = 'output_video.mp4'
# Đường dẫn để lưu ảnh
output_image_path = 'output_image.jpg'

# Mở video
video_capture = cv2.VideoCapture(input_video_path)

# Kiểm tra nếu video được mở thành công
if not video_capture.isOpened():
    print("Error: Could not open video.")
    exit()

# Đọc và lưu 16 frame liên tiếp
frames = []
frame_count = 0

while frame_count < 16:
    ret, frame = video_capture.read()
    if not ret:
        break
    frames.append(frame)
    frame_count += 1

# Lưu video mới
if frames:
    height, width, layers = frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, 30.0, (width, height))

    for frame in frames:
        out.write(frame)

    out.release()

# Lưu một ảnh từ các frame
if frames:
    cv2.imwrite(output_image_path, frames[0])  # Lưu ảnh từ frame đầu tiên

# Giải phóng tài nguyên
video_capture.release()
cv2.destroyAllWindows()

print("Video and image saved successfully.")

Video and image saved successfully.


In [17]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import cv2
import numpy as np
# from torchreid.models import osnet_x0_25  # OSNet model
import os

# ================== 1. Định nghĩa mô hình C3D ==================
class C3D(nn.Module):
    def __init__(self):
        super(C3D, self).__init__()
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 3, 3), stride=1, padding=1)
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), stride=1, padding=1)
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3, 3, 3), stride=1, padding=1)
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3, 3, 3), stride=1, padding=1)
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.fc6 = nn.Linear(256 * 4 * 4 * 4, 256)  # Flatten output

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.pool3(torch.relu(self.conv3b(torch.relu(self.conv3a(x)))))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc6(x)
        return x

# ================== 2. Kết hợp C3D và OSNet ==================
class ReIDModel(nn.Module):
    def __init__(self):
        super(ReIDModel, self).__init__()
        self.osnet = osnet_x0_25(pretrained=True)
        self.c3d = C3D()
        self.fusion = nn.Linear(512 + 256, 256)  # Ghép OSNet (512) và C3D (256)

    def forward(self, img, video_clip):
        osnet_feat = self.osnet(img)  # OSNet trích xuất đặc trưng không gian
        c3d_feat = self.c3d(video_clip)  # C3D trích xuất đặc trưng thời gian
        combined_feat = torch.cat((osnet_feat, c3d_feat), dim=1)  # Ghép feature
        final_feat = self.fusion(combined_feat)  # Đưa qua mạng fully connected
        return final_feat

# ================== 3. Load model ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reid_model = ReIDModel().to(device)
reid_model.eval()

# ================== 4. Chuẩn bị dữ liệu ==================
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 256)),
    transforms.ToTensor()
])

# Load một ảnh đơn giản
img = cv2.imread("/content/output_image.jpg")  # Thay ảnh bằng ảnh có sẵn
img_resized = transform(img).unsqueeze(0).to(device)

# Load 16 frames từ một video
cap = cv2.VideoCapture("/content/output_video.mp4")  # Thay bằng video có sẵn
frame_buffer = []

while len(frame_buffer) < 16:
    ret, frame = cap.read()
    if not ret:
        break
    frame_resized = cv2.resize(frame, (112, 112))
    frame_tensor = torch.tensor(frame_resized).permute(2, 0, 1).unsqueeze(0).float().to(device)
    frame_buffer.append(frame_tensor)

cap.release()

# Nếu chưa đủ 16 frames, lặp lại frame cuối
while len(frame_buffer) < 16:
    frame_buffer.append(frame_buffer[-1])

# Ghép lại thành tensor 3D (1, 3, 16, 112, 112)
# video_clip = torch.stack(frame_buffer).permute(1, 0, 2, 3)  # (C, T, H, W)
# video_clip = video_clip.unsqueeze(0).to(device)
# Chuyển frame_buffer thành tensor 5D: (B=1, C=3, T=16, H=112, W=112)
video_clip = torch.cat(frame_buffer, dim=0)  # (16, 3, 112, 112)
video_clip = video_clip.permute(1, 0, 2, 3).unsqueeze(0).to(device)  # (1, 3, 16, 112, 112)


# ================== 5. Chạy model Re-ID ==================
feature_vector = reid_model(img_resized, video_clip)

print("Feature vector shape:", feature_vector.shape)  # Kết quả mong đợi: (1, 256)


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x200704 and 16384x256)